# Selección de Variables y Comparación de Modelos
### Estadística Computacional para la Toma de Decisiones

**Magíster en Ciencia de Datos e Inteligencia Artificial**
Universidad Andrés Bello

*Maidana, J.P. (2026). Apunte — versión Jupyter Notebook.*

---

Versión **ejecutable** del apunte. Cada método (stepwise, AIC/BIC, validación cruzada) se aplica sobre un mismo conjunto de datos para que puedas correr el código, ver los resultados y modificarlos.

> 💡 Busca los comentarios **`👉 EXPERIMENTA`**: marcan parámetros que vale la pena cambiar para ver cómo reacciona la selección.

**Una nota sobre los datos:** el apunte genera el dataset con `make_regression(...)`. Aquí le agregamos `noise=25` (el valor por defecto es `0`, que produce un ajuste perfecto sin nada que sobreajustar ni que penalizar) y `shuffle=False` (para que las 5 variables informativas sean exactamente `X1`–`X5`). Así los métodos ilustran de verdad los conceptos. Sólo se usan `numpy`, `pandas`, `matplotlib`, `scikit-learn` y `statsmodels`.

## Configuración del entorno

Importamos librerías, fijamos semillas y definimos un estilo de gráficos. **Ejecuta esta celda primero.**

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

np.random.seed(42)

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

# Paleta coherente
AZUL    = "#2c7fb8"
VERDE   = "#41ab5d"
NARANJA = "#fe9929"
ROJO    = "#e34a33"

print("Entorno listo ✔")

### Utilidades para los diagramas conceptuales

Funciones de dibujo (`caja`, `flecha`) para reproducir en `matplotlib` los diagramas de bloques del apunte. No es necesario entenderlas en detalle.

In [ ]:
def caja(ax, x, y, w, h, texto, color, fontsize=10):
    # Caja redondeada con texto centrado.
    box = FancyBboxPatch((x, y), w, h,
                         boxstyle="round,pad=0.02,rounding_size=0.12",
                         linewidth=1.6, edgecolor="#37474f",
                         facecolor=color, alpha=0.92)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, texto, ha="center", va="center", fontsize=fontsize, zorder=5)

def flecha(ax, p1, p2, texto="", dy=0.12):
    # Flecha de p1 a p2 con etiqueta opcional.
    ax.annotate("", xy=p2, xytext=p1,
                arrowprops=dict(arrowstyle="-|>", lw=2.2, color="#37474f"))
    if texto:
        mx, my = (p1[0]+p2[0])/2, (p1[1]+p2[1])/2
        ax.text(mx, my + dy, texto, ha="center", va="bottom", fontsize=9.5, style="italic")

---
# 1. Introducción

Cuando construimos modelos estadísticos, enfrentamos una pregunta fundamental: ¿qué variables incluir? Rara vez la respuesta es obvia. Incluir **todas** las variables disponibles puede llevar a *overfitting*, donde el modelo memoriza el ruido de los datos de entrenamiento y generaliza mal. Incluir **muy pocas** variables puede llevar a *underfitting*, donde el modelo es demasiado simple para capturar patrones importantes.

La **selección de variables** es el proceso de identificar el subconjunto óptimo de predictores. No es sólo una cuestión técnica sino conceptual: queremos modelos **parsimoniosos** —simples de interpretar y comunicar— pero lo suficientemente complejos para capturar las relaciones importantes.

Existen múltiples enfoques. Los métodos *stepwise* (forward, backward y stepwise) construyen o deconstruyen modelos añadiendo o eliminando variables secuencialmente. Los criterios de información como **AIC** y **BIC** proporcionan métricas objetivas para comparar modelos, penalizando la complejidad. La **validación cruzada** estima cuán bien generalizará un modelo a datos nuevos, siendo fundamental para evitar overfitting.

> ### 📝 Lo que dominarás al finalizar
> Entenderás la tensión entre complejidad y parsimonia. Sabrás implementar métodos *forward*, *backward* y *stepwise*. Podrás usar criterios AIC/BIC para comparación objetiva de modelos. Dominarás la validación cruzada para evaluación robusta. Reconocerás los peligros del overfitting y cómo evitarlos. Desarrollarás un enfoque sistemático y reflexivo para construir modelos parsimoniosos y efectivos.

---
# 2. El Problema de Selección de Variables

## 2.1 Parsimonia vs Complejidad

El **principio de parsimonia** (la navaja de Occam) sugiere que entre modelos con desempeño similar, preferimos el más simple: un modelo con 3 variables que predice igual de bien que uno con 15 es preferible porque es más fácil de entender, comunicar y mantener.

Pero la simplicidad tiene un límite. Un modelo demasiado simple no captura patrones importantes. La clave es encontrar el **punto óptimo**: el modelo más simple que aún captura las relaciones esenciales.

In [ ]:
# Diagrama conceptual: error de entrenamiento vs prueba según complejidad
x = np.linspace(0.5, 9.5, 200)
err_train = 5 * np.exp(-0.3 * x) + 0.5
err_test  = 2.5 * np.exp(-0.3 * x) + 0.3 * (x - 5)**2 / 5 + 1

fig, ax = plt.subplots()
ax.plot(x, err_train, color=AZUL, lw=2.5, label="Error en entrenamiento")
ax.plot(x, err_test,  color=ROJO, lw=2.5, label="Error en prueba")

# Punto óptimo (mínimo del error de prueba)
i_opt = np.argmin(err_test)
ax.axvline(x[i_opt], color=VERDE, ls="--", lw=2)
ax.scatter([x[i_opt]], [err_test[i_opt]], color=VERDE, s=70, zorder=5)
ax.text(x[i_opt], -0.4, "Óptimo", color=VERDE, ha="center", weight="bold")

ax.annotate("Underfitting", xy=(1.5, 5.2), fontsize=10, color="#555", ha="center")
ax.annotate("Overfitting",  xy=(8, 5.2),  fontsize=10, color="#555", ha="center")
ax.set_xlabel("Complejidad del modelo  (simple → complejo)")
ax.set_ylabel("Error")
ax.set_title("La tensión fundamental: sesgo vs varianza", weight="bold")
ax.set_xticks([]); ax.set_ylim(0, 6)
ax.legend()
plt.tight_layout(); plt.show()

## 2.2 Consecuencias de mala selección

**Incluir variables irrelevantes** aumenta la varianza de las predicciones sin reducir el sesgo. El modelo se vuelve más sensible al ruido, los coeficientes de las variables relevantes se vuelven menos precisos (multicolinealidad), y el modelo es más difícil de interpretar.

**Omitir variables importantes** introduce sesgo: el modelo predice sistemáticamente mal porque falta información crucial. Los coeficientes incluidos pueden estar sesgados si están correlacionados con las omitidas (*sesgo de variable omitida*), y el modelo pierde poder predictivo.

## 2.3 Desafíos fundamentales

El número de modelos posibles crece **exponencialmente**: con $p$ variables hay $2^p$ subconjuntos. Con 10 variables hay 1,024 modelos; con 20 hay más de un millón. Es computacionalmente imposible probarlos todos.

In [ ]:
# El espacio de modelos crece como 2^p
p = np.arange(1, 21)
n_modelos = 2.0 ** p

fig, ax = plt.subplots()
ax.semilogy(p, n_modelos, "o-", color=AZUL)
ax.scatter([10, 20], [2**10, 2**20], color=ROJO, zorder=5, s=60)
ax.annotate("p=10 → 1,024 modelos", xy=(10, 2**10), xytext=(2.5, 2**13),
            arrowprops=dict(arrowstyle="->", color="#555"), fontsize=9.5)
ax.annotate("p=20 → 1,048,576 modelos", xy=(20, 2**20), xytext=(8.5, 2**17),
            arrowprops=dict(arrowstyle="->", color="#555"), fontsize=9.5)
ax.set_xlabel("Número de variables ($p$)")
ax.set_ylabel("Modelos posibles  $2^p$  (escala log)")
ax.set_title("Por qué no podemos probar todos los subconjuntos", weight="bold")
ax.set_xticks(range(1, 21, 2))
plt.tight_layout(); plt.show()

Además, las variables pueden **interactuar** de formas complejas (una variable irrelevante por sí sola puede importar en presencia de otra), y los datos tienen **ruido**: en muestras finitas, variables irrelevantes pueden parecer significativas por casualidad, especialmente al probar muchas.

> ### ❗ El rol del conocimiento del dominio
> La selección de variables **nunca** debe ser puramente mecánica. Los métodos estadísticos son herramientas, no sustitutos del pensamiento crítico.
>
> El conocimiento del dominio te dice qué variables tienen sentido teórico. Si una variable es significativa estadísticamente pero no tiene justificación causal, **sospecha**. Si es importante teóricamente pero no significativa, **investiga** por qué antes de eliminarla.
>
> Los mejores modelos combinan evidencia estadística con razonamiento substantivo. Usa los métodos como guías, pero siempre pregunta si los resultados tienen sentido en el contexto del problema.

---
# 3. Métodos Stepwise

Los métodos *stepwise* son enfoques algorítmicos que construyen o deconstruyen modelos secuencialmente, añadiendo o eliminando variables una a la vez.

## 3.1 Forward Selection (hacia adelante)

Comienza con un modelo vacío (sólo intercepto) y **añade** variables una a la vez. En cada paso añade la variable que más mejora el modelo (típicamente menor valor $p$ o menor AIC).

**Algoritmo:**
1. Comienza con modelo vacío: $y = \beta_0$.
2. Para cada variable no incluida, ajusta un modelo temporal añadiéndola.
3. Elige la variable con mayor mejora (menor $p$ o menor AIC).
4. Si la mejora es significativa (ej. $p < 0.05$), añádela permanentemente.
5. Repite 2–4 hasta que ninguna variable mejore significativamente.

**Ventajas:** simple, computacionalmente eficiente, útil con muchas variables y pocas observaciones.
**Desventajas:** una vez añadida, una variable nunca se elimina; no considera todas las combinaciones; sensible a correlaciones entre predictores.

## 3.2 Backward Elimination (hacia atrás)

Comienza con el modelo **completo** (todas las variables) y **elimina** la menos importante en cada paso.

**Algoritmo:**
1. Comienza con todas las variables incluidas.
2. Identifica la variable con menor contribución (mayor valor $p$).
3. Si es no significativa (ej. $p > 0.10$), elimínala.
4. Repite 2–3 hasta que todas las restantes sean significativas.

**Ventajas:** considera todas las variables desde el inicio, capturando potencialmente *supresores* (variables que sólo importan en presencia de otras).
**Desventajas:** requiere $n > p$; una vez eliminada, una variable no se reconsidera; costoso con muchas variables.

## 3.3 Stepwise Selection (por pasos)

Híbrido que combina forward y backward: añade variables como *forward*, pero después de cada adición verifica si alguna variable previa se volvió redundante y debe eliminarse.

**Ventajas:** más flexible; puede corregir decisiones anteriores.
**Desventajas:** aún no considera todas las combinaciones; puede ciclar si los criterios no están bien definidos (requiere umbrales de entrada y salida **distintos**).

### Generación del dataset

Usaremos este mismo conjunto de datos para todos los métodos del apunte.

In [ ]:
import statsmodels.api as sm
from sklearn.datasets import make_regression

# 200 observaciones, 10 variables, sólo 5 informativas + ruido.
# noise=25  -> hay algo que ajustar y que penalizar (el default 0 da R2=1).
# shuffle=False -> las informativas son exactamente X1..X5.
X, y = make_regression(n_samples=200, n_features=10, n_informative=5,
                       noise=25, shuffle=False, random_state=42)

feature_names = [f"X{i+1}" for i in range(10)]
X_df = pd.DataFrame(X, columns=feature_names)

print(f"Dataset: {X_df.shape[0]} observaciones, {X_df.shape[1]} variables")
print("Por construcción: X1–X5 son informativas, X6–X10 son ruido.")
X_df.head()

### Forward selection

In [ ]:
def forward_selection(X, y, significance_level=0.05):
    '''Forward selection basado en p-values.'''
    best_features = []
    remaining_features = list(X.columns)

    print("=" * 60)
    print("FORWARD SELECTION")
    print("=" * 60)

    while remaining_features:
        p_values = []
        for feature in remaining_features:                    # probar cada candidata
            X_test = sm.add_constant(X[best_features + [feature]])
            model = sm.OLS(y, X_test).fit()
            p_values.append((feature, model.pvalues[feature]))

        p_values.sort(key=lambda t: t[1])                     # mejor = menor p-value
        best_feature, best_p = p_values[0]

        if best_p < significance_level:
            best_features.append(best_feature)
            remaining_features.remove(best_feature)
            print(f"Añadida:   {best_feature:4s} (p-value: {best_p:.4f})")
        else:
            break

    print(f"\nVariables finales: {best_features}")
    return best_features

forward_vars = forward_selection(X_df, y)

### Backward elimination

In [ ]:
def backward_elimination(X, y, significance_level=0.10):
    '''Backward elimination basado en p-values.'''
    features = list(X.columns)

    print("=" * 60)
    print("BACKWARD ELIMINATION")
    print("=" * 60)

    while len(features) > 0:
        model = sm.OLS(y, sm.add_constant(X[features])).fit()
        p_values = model.pvalues.iloc[1:]                     # excluir la constante
        max_p_value = p_values.max()

        if max_p_value > significance_level:
            excluded = p_values.idxmax()
            features.remove(excluded)
            print(f"Eliminada: {excluded:4s} (p-value: {max_p_value:.4f})")
        else:
            break

    print(f"\nVariables finales: {features}")
    return features

backward_vars = backward_elimination(X_df, y)

### Stepwise selection

El apunte describe el método híbrido pero no lo implementa. Aquí va una versión completa, con umbrales de **entrada** (0.05) y **salida** (0.10) distintos para evitar ciclos.

In [ ]:
def stepwise_selection(X, y, entrada=0.05, salida=0.10, max_iter=100):
    '''Stepwise: combina un paso forward y un paso backward en cada iteración.
       entrada < salida evita que una variable recién añadida se elimine de inmediato.'''
    incluidas = []

    print("=" * 60)
    print("STEPWISE SELECTION")
    print("=" * 60)

    for _ in range(max_iter):
        cambio = False

        # --- paso forward: intentar añadir la mejor variable externa ---
        externas = [c for c in X.columns if c not in incluidas]
        if externas:
            p_nuevas = []
            for c in externas:
                m = sm.OLS(y, sm.add_constant(X[incluidas + [c]])).fit()
                p_nuevas.append((c, m.pvalues[c]))
            p_nuevas.sort(key=lambda t: t[1])
            cand, p_cand = p_nuevas[0]
            if p_cand < entrada:
                incluidas.append(cand); cambio = True
                print(f"  + Añadida:   {cand:4s} (p: {p_cand:.4f})")

        # --- paso backward: intentar eliminar la peor variable incluida ---
        if incluidas:
            m = sm.OLS(y, sm.add_constant(X[incluidas])).fit()
            p = m.pvalues.iloc[1:]
            if p.max() > salida:
                peor = p.idxmax()
                incluidas.remove(peor); cambio = True
                print(f"  - Eliminada: {peor:4s} (p: {p.max():.4f})")

        if not cambio:
            break

    print(f"\nVariables finales: {incluidas}")
    return incluidas

stepwise_vars = stepwise_selection(X_df, y)

Los tres métodos convergen al mismo conjunto (`X1`–`X5`), que son exactamente las variables informativas: una señal de que la selección está funcionando bien. En problemas reales con multicolinealidad o señales débiles, los métodos pueden discrepar.

> ### 🚨 Limitaciones de los métodos stepwise
> **Sobreajuste de valores $p$:** al probar múltiples variables secuencialmente, los valores $p$ dejan de ser válidos (múltiples comparaciones inflan el error tipo I).
>
> **No óptimo global:** son *greedy* (avaros): toman la mejor decisión local en cada paso, sin garantizar el mejor modelo global.
>
> **Ignoran interacciones:** si $X_1$ y $X_2$ sólo importan juntas, *forward* puede no encontrarlas.
>
> **Sensibles al orden:** con multicolinealidad, el orden en que se prueban afecta cuál se selecciona.
>
> Son útiles como herramientas **exploratorias**, especialmente con muchas variables candidatas. No los uses mecánicamente sin reflexión.

---
# 4. Criterios de Información: AIC y BIC

Los criterios de información comparan modelos con distinto número de variables, balanceando ajuste y complejidad.

## 4.1 El principio fundamental

Tanto **AIC** (Akaike) como **BIC** (Bayesian) recompensan modelos que ajustan bien (alta verosimilitud) pero penalizan complejidad (muchos parámetros):

$$ \text{Criterio} = -2 \log(L) + \text{penalización} \times k $$

donde $L$ es la verosimilitud y $k$ el número de parámetros. **Valores más bajos = mejores modelos.**

## 4.2 AIC

$$ \text{AIC} = -2 \log(L) + 2k = n \log(\text{RSS}/n) + 2k $$

Penaliza cada parámetro con un costo fijo de **2**. Penalización suave → favorece modelos ligeramente más complejos. Diseñado para **minimizar el error de predicción** → apropiado cuando el objetivo es predicción.

## 4.3 BIC

$$ \text{BIC} = -2 \log(L) + k \log(n) = n \log(\text{RSS}/n) + k \log(n) $$

La penalización es $\log(n)$ en lugar de 2. Para $n > 7$, $\log(n) > 2$ → BIC penaliza más fuerte que AIC, y la penalización **aumenta con el tamaño de muestra**. Diseñado para **encontrar el modelo verdadero** → apropiado para inferencia e interpretación.

## 4.4 Comparación e interpretación

| Aspecto | AIC | BIC |
|---|---|---|
| Penalización | $2k$ (fija) | $k \log(n)$ (aumenta con $n$) |
| Modelos favorecidos | Ligeramente más complejos | Más parsimoniosos |
| Objetivo | Minimizar error de predicción | Encontrar modelo verdadero |
| Uso recomendado | Predicción | Inferencia, interpretación |
| Muestras grandes | Puede sobreajustar | Más conservador |

**Interpretación de diferencias** $\Delta$ en AIC o BIC:

| $\Delta$ | Interpretación |
|---|---|
| $\Delta < 2$ | Diferencia no sustancial; modelos esencialmente equivalentes |
| $2 \le \Delta < 6$ | Evidencia positiva a favor del menor criterio |
| $6 \le \Delta < 10$ | Evidencia fuerte |
| $\Delta \ge 10$ | Evidencia muy fuerte |

> ### 🧪 Ejemplo (ilustrativo) — Comparación con AIC/BIC
> Tres modelos para predecir ventas:
> - **Modelo 1:** 3 variables, AIC = 450.2, BIC = 462.8
> - **Modelo 2:** 5 variables, AIC = 448.5, BIC = 466.3
> - **Modelo 3:** 8 variables, AIC = 447.8, BIC = 472.1
>
> **Según AIC:** Modelo 3 es mejor (menor AIC). Diferencias: $\Delta_{3-2}=0.7$ (mínima), $\Delta_{3-1}=2.4$ (leve).
> **Según BIC:** Modelo 1 es mejor (menor BIC). Diferencias: $\Delta_{1-2}=3.5$ (positiva), $\Delta_{1-3}=9.3$ (fuerte).
>
> **Conclusión:** AIC favorece el modelo más complejo (mejor predicción), BIC el más parsimonioso (mejor interpretación). La elección depende del objetivo.

Apliquémoslo sobre **nuestros datos** con tres modelos anidados:

In [ ]:
# Comparar modelos con AIC/BIC sobre el dataset real
models = {
    "Modelo 1": ["X1", "X2", "X3"],
    "Modelo 2": ["X1", "X2", "X3", "X4", "X5"],
    "Modelo 3": ["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"],
}

resultados = []
for nombre, variables in models.items():
    modelo = sm.OLS(y, sm.add_constant(X_df[variables])).fit()
    resultados.append({
        "Modelo": nombre, "Variables": len(variables),
        "R2": round(modelo.rsquared, 4), "R2_aj": round(modelo.rsquared_adj, 4),
        "AIC": round(modelo.aic, 1), "BIC": round(modelo.bic, 1),
    })

df_res = pd.DataFrame(resultados)
print("COMPARACIÓN DE MODELOS:")
print(df_res.to_string(index=False))

print(f"\nMejor según R² ajustado: {df_res.loc[df_res['R2_aj'].idxmax(), 'Modelo']}")
print(f"Mejor según AIC:         {df_res.loc[df_res['AIC'].idxmin(), 'Modelo']}")
print(f"Mejor según BIC:         {df_res.loc[df_res['BIC'].idxmin(), 'Modelo']}")

El **Modelo 2** (las 5 variables informativas) gana en AIC *y* BIC: el Modelo 1 subajusta (le faltan informativas) y el Modelo 3 agrega ruido que ambos criterios penalizan. Veamos qué pasa al ir agregando variables **una a una** (en el orden de utilidad que encontró *forward selection*):

In [ ]:
# AIC y BIC al aumentar el número de variables (orden de utilidad de forward)
orden = forward_vars + [c for c in X_df.columns if c not in forward_vars]

aic_vals, bic_vals = [], []
for k in range(1, len(orden) + 1):
    m = sm.OLS(y, sm.add_constant(X_df[orden[:k]])).fit()
    aic_vals.append(m.aic); bic_vals.append(m.bic)

ks = list(range(1, len(orden) + 1))
k_aic = ks[int(np.argmin(aic_vals))]
k_bic = ks[int(np.argmin(bic_vals))]

fig, ax = plt.subplots()
ax.plot(ks, aic_vals, "o-", color=AZUL, label="AIC")
ax.plot(ks, bic_vals, "s-", color=NARANJA, label="BIC")
ax.axvline(k_aic, color=AZUL, ls=":", alpha=0.7)
ax.axvline(k_bic, color=NARANJA, ls=":", alpha=0.7)
ax.scatter([k_aic], [min(aic_vals)], color=AZUL, s=80, zorder=5)
ax.scatter([k_bic], [min(bic_vals)], color=NARANJA, s=80, zorder=5)
ax.set_xlabel("Número de variables en el modelo")
ax.set_ylabel("Criterio de información")
ax.set_title("AIC y BIC penalizan la complejidad (mínimo en 5 variables)", weight="bold")
ax.set_xticks(ks)
ax.legend()
plt.tight_layout(); plt.show()

print(f"AIC se minimiza con {k_aic} variables; BIC con {k_bic}.")
print("Ambos identifican las 5 informativas; nota cómo BIC sube más rápido (penaliza más).")

---
# 5. Validación Cruzada

Técnica fundamental para evaluar qué tan bien un modelo **generalizará** a datos nuevos.

## 5.1 El problema con train/test simple

Una división simple (80% train / 20% test) funciona, pero tiene limitaciones: la evaluación depende de qué observaciones cayeron en test (con mala suerte, no es representativo) y se desperdicia información (sólo 80% se usa para entrenar).

## 5.2 K-Fold Cross-Validation

Divide los datos en $k$ grupos (*folds*) de tamaño similar:
1. Divide en $k$ folds.
2. Para cada fold $i$: úsalo como test, entrena con los otros $k-1$, evalúa y guarda la métrica.
3. Promedia las $k$ métricas.

In [ ]:
# Diagrama: 5-Fold Cross-Validation
k = 5
fig, ax = plt.subplots(figsize=(9, 4))
ax.set_xlim(-1.2, k + 0.2); ax.set_ylim(-k - 0.3, 1); ax.axis("off")

for i in range(k):           # fila i: el bloque i es test (rojo), el resto train (azul)
    for j in range(k):
        color = ROJO if j == i else AZUL
        etiqueta = "Test" if j == i else "Train"
        ax.add_patch(plt.Rectangle((j, -i - 0.8), 1, 0.7, facecolor=color,
                                    edgecolor="white", lw=2, alpha=0.85))
        ax.text(j + 0.5, -i - 0.45, etiqueta, ha="center", va="center",
                color="white", fontsize=9, weight="bold")
    ax.text(-0.15, -i - 0.45, f"Iter {i+1}", ha="right", va="center", fontsize=9)

ax.text(k/2, 0.4, "5-Fold Cross-Validation", ha="center", fontsize=13, weight="bold")
plt.tight_layout(); plt.show()

Típicamente $k=5$ o $k=10$. Con $k=5$ usas 80% para entrenar en cada iteración pero obtienes **5** estimaciones (no 1), usando todos los datos. Con $k=10$ usas 90%, maximizando el uso de datos.

## 5.3 Leave-One-Out (LOOCV)

Caso extremo con $k=n$: en cada iteración entrenas con $n-1$ observaciones y pruebas en la restante.

**Ventajas:** usa el máximo de datos; evaluación casi insesgada.
**Desventajas:** costoso ($n$ modelos); alta varianza (conjuntos de train casi idénticos); inapropiado para datasets grandes. Útil sobre todo en muestras pequeñas ($n < 100$).

## 5.4 Implementación y uso

In [ ]:
from sklearn.model_selection import cross_val_score, KFold, train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

modelo = LinearRegression()
kfold = KFold(n_splits=5, shuffle=True, random_state=42)   # 👉 EXPERIMENTA: prueba n_splits=10

scores_r2   =  cross_val_score(modelo, X_df, y, cv=kfold, scoring="r2")
scores_rmse = -cross_val_score(modelo, X_df, y, cv=kfold,
                               scoring="neg_root_mean_squared_error")

print("=" * 60)
print("K-FOLD CROSS-VALIDATION (k=5)")
print("=" * 60)
for i, s in enumerate(scores_r2, 1):
    print(f"  Fold {i}:  R² = {s:.4f}")
print(f"\nR²   promedio: {scores_r2.mean():.4f}  (± {scores_r2.std():.4f})")
print(f"RMSE promedio: {scores_rmse.mean():.4f}  (± {scores_rmse.std():.4f})")

# Comparación con train/test simple (una sola división)
X_tr, X_te, y_tr, y_te = train_test_split(X_df, y, test_size=0.2, random_state=42)
modelo.fit(X_tr, y_tr)
y_pred = modelo.predict(X_te)
print("\n" + "=" * 60)
print("TRAIN/TEST SIMPLE (80/20)")
print("=" * 60)
print(f"R²:   {r2_score(y_te, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_te, y_pred)):.4f}")
print("\nLa CV entrega 5 estimaciones (y su variabilidad), no una sola foto.")

In [ ]:
# Visualizar la variabilidad entre folds
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(1, 6), scores_r2, color=AZUL, alpha=0.85)
ax.axhline(scores_r2.mean(), color=ROJO, ls="--", lw=2,
           label=f"Promedio = {scores_r2.mean():.3f}")
ax.set_ylim(0.85, 1.0)
ax.set_xlabel("Fold"); ax.set_ylabel("R²")
ax.set_title("R² por fold: cada división da un resultado distinto", weight="bold")
ax.legend()
plt.tight_layout(); plt.show()

## 5.5 Validación cruzada para selección de variables

Una aplicación crucial: en lugar de elegir el modelo por AIC/BIC calculados sobre **todos** los datos (que puede sobreajustar), comparamos los modelos candidatos por su **error de CV**. El que mejor generaliza es el preferido.

In [ ]:
# Comparar los modelos candidatos por CV (estimación honesta de generalización)
print("Comparación de modelos por validación cruzada (R²):\n")
for nombre, variables in models.items():
    sc = cross_val_score(LinearRegression(), X_df[variables], y, cv=kfold, scoring="r2")
    print(f"  {nombre} ({len(variables)} vars): CV R² = {sc.mean():.4f}  (± {sc.std():.4f})")

print("\nEl Modelo 2 generaliza mejor: coincide con lo que indicaron AIC y BIC.")

> ### ❗ Validación cruzada: la mejor práctica
> La CV debe ser tu **estándar de oro** para evaluación de modelos:
> - **Selección de modelos:** compara subconjuntos de variables por CV; el de mejor CV generalizará mejor.
> - **Tuning de hiperparámetros:** en regularización (Ridge, Lasso), usa CV para elegir el parámetro de penalización.
> - **Evaluación final:** reporta métricas de CV junto con (o en lugar de) las de train/test simple.
>
> La inversión computacional vale la pena: CV da estimaciones más robustas y reduce el riesgo de decidir según una división afortunada o desafortunada.

---
# 6. Evitando Overfitting

El *overfitting* es uno de los problemas más insidiosos: un modelo puede verse perfecto en entrenamiento pero fallar en datos nuevos.

## 6.1 Señales de overfitting

- **Gran diferencia train vs test:** $R^2$ de 0.95 en train pero 0.60 en test → overfitting severo.
- **Muchas variables relativas a observaciones:** si $p$ se acerca a $n$, estás en zona de peligro. Regla útil: 10–15 observaciones por variable.
- **Coeficientes con magnitudes muy grandes:** sugieren ajuste al ruido.
- **Multicolinealidad alta:** VIF muy altos indican efectos redundantes.
- **Modelo complejo con mejora marginal:** si 5 variables más sólo suben $R^2$ de 0.82 a 0.84, probablemente sobreajustas.

La primera señal —la brecha train/test— se ve con claridad cuando el número de variables se acerca al de observaciones. Lo demostramos con un escenario construido para ello: $n=40$ observaciones, 5 variables informativas y 35 de **ruido**. Empezamos con las informativas y vamos agregando ruido, promediando el $R^2$ sobre 40 particiones aleatorias para obtener una curva limpia:

In [ ]:
# Escenario construido para EXPONER el overfitting: muchas variables, pocas observaciones.
# 5 informativas (V1–V5) + 35 de ruido (V6–V40); sólo 40 observaciones de entrenamiento.
from sklearn.model_selection import ShuffleSplit

X_of, y_of = make_regression(n_samples=60, n_features=40, n_informative=5,
                             noise=20, shuffle=False, random_state=1)
cols_of = [f"V{i+1}" for i in range(40)]
X_of = pd.DataFrame(X_of, columns=cols_of)

ks = list(range(1, 40, 2))                                   # 1, 3, 5, ..., 39 variables
splits = ShuffleSplit(n_splits=40, train_size=40, random_state=0)  # 👉 EXPERIMENTA: train_size

train_r2, test_r2 = [], []
for k in ks:
    cols = cols_of[:k]                                        # V1..Vk (las 5 primeras informativas)
    tr, te = [], []
    for i_tr, i_te in splits.split(X_of):                    # promediar sobre 40 particiones
        m = LinearRegression().fit(X_of[cols].iloc[i_tr], y_of[i_tr])
        tr.append(r2_score(y_of[i_tr], m.predict(X_of[cols].iloc[i_tr])))
        te.append(r2_score(y_of[i_te], m.predict(X_of[cols].iloc[i_te])))
    train_r2.append(np.mean(tr)); test_r2.append(np.mean(te))

test_plot = np.maximum(test_r2, -0.3)                        # recortar el colapso para la escala

fig, ax = plt.subplots()
ax.plot(ks, train_r2,  "o-", color=AZUL, label="R² en entrenamiento")
ax.plot(ks, test_plot, "s-", color=ROJO, label="R² en prueba (promedio 40 particiones)")
ax.axvline(5, color=VERDE, ls="--", lw=2, label="Complejidad real (5 variables)")
ax.set_ylim(-0.35, 1.05)
ax.set_xlabel("Número de variables en el modelo  (n = 40 observaciones)")
ax.set_ylabel("R²")
ax.set_title("Overfitting: el train sube siempre; el test colapsa cuando p → n", weight="bold")
ax.annotate("el test se desploma\ncuando p ≈ n", xy=(37, -0.3), xytext=(25, -0.1),
            arrowprops=dict(arrowstyle="->", color=ROJO), color=ROJO, fontsize=9)
ax.legend(loc="lower left", fontsize=9)
plt.tight_layout(); plt.show()

print(f"Con 5 variables (la complejidad real), el test alcanza su máximo: R² ≈ {max(test_r2):.2f}.")
print(f"Con 39 variables el train es casi perfecto pero el test es catastrófico: R² ≈ {test_r2[-1]:.0f}.")

## 6.2 Estrategias de prevención

1. **Más datos:** la forma más efectiva. Con suficientes observaciones, modelos complejos se vuelven sostenibles.
2. **Selección criteriosa:** no incluyas variables sólo por ser significativas; pregunta si tienen sentido causal. Usa stepwise, AIC/BIC, CV.
3. **Validación cruzada:** si el error de CV es mucho mayor que el de train, sobreajustas.
4. **División adecuada:** mantén un test completamente separado; no ajustes hiperparámetros con test.
5. **Regularización:** Ridge y Lasso penalizan magnitudes de coeficientes, especialmente útiles cuando $p$ es grande.
6. **Parsimonia:** entre modelos similares, elige el más simple.

## 6.3 El compromiso sesgo–varianza revisitado

El overfitting es una manifestación del compromiso sesgo–varianza. Modelos simples tienen **alto sesgo** (sistemáticamente equivocados) pero **baja varianza** (consistentes). Modelos complejos tienen **bajo sesgo** pero **alta varianza** (muy sensibles a los datos de entrenamiento).

$$ \text{Error Total} = \text{Sesgo}^2 + \text{Varianza} + \text{Ruido Irreducible} $$

No puedes eliminar el ruido irreducible. Tu objetivo es minimizar la suma de sesgo y varianza: reducir varianza (simplificar) típicamente aumenta sesgo, y viceversa. La clave es el **balance óptimo**.

> ### 🚨 Data snooping: un peligro sutil
> *Data snooping* (husmear los datos) ocurre cuando dejas que los datos de test influyan en tus decisiones de modelamiento.
>
> **Ejemplos:** ajustas un modelo, ves el error en test, decides añadir una variable y vuelves a mirar test (usaste test para decidir); pruebas muchas transformaciones eligiendo la mejor en test; eliminas outliers tras ver que mejora el test.
>
> **La solución:** usa validación cruzada en *train* para **todas** las decisiones. Reserva test sólo para la evaluación final. Mejor aún, usa **tres** divisiones: train, validación (para decisiones) y test (para evaluación honesta).
>
> Es fácil caer en *data snooping* inadvertidamente. Sé disciplinado con tus datos de evaluación.

---
# 7. Estrategia Integral de Selección

Juntemos todo en una estrategia comprensiva.

## 7.1 Workflow recomendado

1. **Exploración y preparación:** trata faltantes, examina correlaciones (multicolinealidad), divide train/test, estandariza si vas a comparar importancia.
2. **Análisis univariado:** examina cada variable individualmente; ¿cuáles correlacionan con $y$? Obtén una lista inicial de candidatas.
3. **Modelo saturado inicial:** ajusta con todas las variables teóricamente relevantes; examina significancia, VIF y bondad de ajuste. Punto de partida, no final.
4. **Selección automática:** aplica stepwise o AIC/BIC como guías exploratorias. ¿Convergen forward, backward y stepwise?
5. **Validación cruzada:** calcula error de CV para los modelos candidatos; estimación robusta de generalización.
6. **Evaluación substantiva:** ¿los modelos tienen sentido teórico? ¿Los signos de los coeficientes son razonables? ¿Faltan variables importantes?
7. **Análisis de sensibilidad:** ¿qué pasa si eliminas observaciones influyentes? ¿El modelo es estable con distintas divisiones?
8. **Modelo final y evaluación en test:** elige, ajusta en train completo y evalúa en test (intacto hasta ahora).

In [ ]:
# Resumen visual del workflow de selección de variables
fig, ax = plt.subplots(figsize=(11, 6))
ax.set_xlim(0, 12); ax.set_ylim(0, 9); ax.axis("off")

izq = ["1. Exploración\ny Preparación", "2. Análisis\nUnivariado",
       "3. Modelo\nSaturado", "4. Selección\nAutomática"]
der = ["5. Validación\nCruzada", "6. Evaluación\nSubstantiva",
       "7. Análisis de\nSensibilidad", "8. Modelo Final\ny Test"]

y0 = 7.5
for i, t in enumerate(izq):
    caja(ax, 0.8, y0 - i*1.9, 3.2, 1.3, t, AZUL, fontsize=9.5)
    if i > 0:
        flecha(ax, (2.4, y0 - (i-1)*1.9), (2.4, y0 - i*1.9 + 1.3))
for i, t in enumerate(der):
    color = VERDE if i < 3 else NARANJA
    caja(ax, 8, y0 - i*1.9, 3.2, 1.3, t, color, fontsize=9.5)
    if i > 0:
        flecha(ax, (9.6, y0 - (i-1)*1.9), (9.6, y0 - i*1.9 + 1.3))

# Conector de la columna izquierda (paso 4) a la derecha (paso 5)
ax.annotate("", xy=(8, y0 + 0.65), xytext=(4, y0 - 3*1.9 + 0.65),
            arrowprops=dict(arrowstyle="-|>", lw=2, color="#37474f",
                            connectionstyle="arc3,rad=-0.25"))

ax.set_title("Workflow de selección de variables", fontsize=13, weight="bold")
plt.tight_layout(); plt.show()

## 7.2 Criterios de decisión

Con múltiples modelos candidatos, considera varios factores: **desempeño predictivo** (¿cuál generaliza mejor según CV?), **parsimonia** (¿cuál es más simple para desempeño comparable?), **interpretabilidad**, **justificación teórica**, **estabilidad** (¿menos sensible a cambios en datos?) y **propósito** (¿predicción pura o comprensión causal?).

No existe un único *"mejor"* modelo: la elección depende del contexto y los objetivos. **Documentar tu proceso de decisión es tan importante como el modelo mismo.**

## 7.3 Documentación y comunicación

Un análisis bien documentado incluye: las **variables consideradas** y por qué; el **proceso de selección** (qué métodos y por qué); los **modelos comparados** (tabla con métricas); la **decisión final** y su justificación; las **limitaciones** (supuestos cumplidos o violados, variables posiblemente omitidas); y el **análisis de sensibilidad**. Esta documentación te protege de críticas de *data snooping* y decisiones arbitrarias.

> ### 💡 Cierre
> La selección de variables es arte y ciencia. Los métodos de este apunte —stepwise, AIC/BIC, validación cruzada— son herramientas poderosas, pero requieren juicio experto. La mejor selección combina evidencia estadística con razonamiento substantivo: usa los métodos como guías, no como sustitutos del pensamiento crítico. Busca modelos que no sólo predigan bien sino que tengan sentido en el contexto del problema. La parsimonia no es sólo elegancia matemática sino necesidad práctica: modelos simples son más robustos, interpretables y útiles para la toma de decisiones. Con estas herramientas y la reflexión cuidadosa, estás equipado para construir modelos verdaderamente útiles en el mundo real.

---
## Referencias bibliográficas

- Akaike, H. (1974). A new look at the statistical model identification. *IEEE Transactions on Automatic Control*, 19(6), 716–723.
- James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). *An introduction to statistical learning* (2nd ed.). Springer.
- Schwarz, G. (1978). Estimating the dimension of a model. *The Annals of Statistics*, 6(2), 461–464.
- Tibshirani, R. (1996). Regression shrinkage and selection via the lasso. *Journal of the Royal Statistical Society: Series B*, 58(1), 267–288.
- Pedregosa, F., et al. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research*, 12, 2825–2830.